# LC #238 — Product of Array Except Self
**Category:** Array / Prefix Products | **Difficulty:** Medium | **Day 1**

---

<div style="padding:15px;border-left:8px solid #667eea;background:#f0f0ff;border-radius:4px;">
<strong>The Problem:</strong> Given an integer array <code>nums</code>, return an array <code>output</code>
where <code>output[i]</code> is the product of all elements in <code>nums</code> except <code>nums[i]</code>.
<strong>No division allowed. Must run in O(n).</strong>
</div>

**Example:**
```
Input:  [1, 2, 3, 4]
Output: [24, 12, 8, 6]

Check: output[0] = 2*3*4 = 24 ✓
       output[1] = 1*3*4 = 12 ✓
       output[2] = 1*2*4 = 8  ✓
       output[3] = 1*2*3 = 6  ✓
```

**Core Insight:** `output[i] = product of everything LEFT of i × product of everything RIGHT of i`. Build each half in a single pass. Two passes total = O(n).

## Mental Models

**1. Left × Right Decomposition**
Any position `i` has exactly two "contributions" — everything to its left, and everything to its right. Compute a left-products array and a right-products array, then multiply element-wise. This is the conceptual foundation.

**2. The Running Product Trick**
You don't need two extra arrays. Use one result array. First pass: fill result[i] with the left product. Second pass: maintain a running right product and multiply into result[i] in-place. O(1) extra space.

**3. Why No Division?**
If you could divide: `total_product / nums[i]`. But — what if nums[i] = 0? Division by zero is undefined. The prefix/suffix approach handles zeros naturally with no special cases. This is why the constraint exists: it forces the elegant solution.

In [ ]:
# Brute Force — O(n²) time, O(n) space
# For each position, multiply all other elements.

def productExceptSelf_brute(nums):
    n = len(nums)
    result = []
    for i in range(n):
        product = 1
        for j in range(n):
            if i != j:
                product *= nums[j]
        result.append(product)
    return result

print(productExceptSelf_brute([1, 2, 3, 4]))  # [24, 12, 8, 6]
print(productExceptSelf_brute([-1, 1, 0, -3, 3]))  # [0, 0, 9, 0, 0]

## Trace: Two-Pass Prefix/Suffix

Input: `[1, 2, 3, 4]`

**Left pass** — result[i] = product of everything to the LEFT:
```
Start: result = [1, 1, 1, 1], prefix = 1

i=0: result[0] = 1  (nothing to left),  prefix = 1 * 1 = 1
i=1: result[1] = 1  (only nums[0]=1),   prefix = 1 * 2 = 2
i=2: result[2] = 2  (nums[0]*nums[1]),  prefix = 2 * 3 = 6
i=3: result[3] = 6  (nums[0..2]),       prefix = 6 * 4 = 24
Result after left pass: [1, 1, 2, 6]
```

**Right pass** — multiply by product of everything to the RIGHT:
```
Start: suffix = 1

i=3: result[3] *= 1  → 6*1  = 6,  suffix = 1 * 4 = 4
i=2: result[2] *= 4  → 2*4  = 8,  suffix = 4 * 3 = 12
i=1: result[1] *= 12 → 1*12 = 12, suffix = 12 * 2 = 24
i=0: result[0] *= 24 → 1*24 = 24, suffix = 24 * 1 = 24
Final: [24, 12, 8, 6]  ✓
```

In [ ]:
# Optimal — O(n) time, O(1) extra space
# Left pass fills result with left products.
# Right pass multiplies in right products using running suffix.

def productExceptSelf(nums: list[int]) -> list[int]:
    n = len(nums)
    result = [1] * n

    # Left pass: result[i] = product of nums[0..i-1]
    prefix = 1
    for i in range(n):
        result[i] = prefix
        prefix *= nums[i]

    # Right pass: multiply result[i] by product of nums[i+1..n-1]
    suffix = 1
    for i in range(n - 1, -1, -1):
        result[i] *= suffix
        suffix *= nums[i]

    return result

# Test
print(productExceptSelf([1, 2, 3, 4]))         # [24, 12, 8, 6]
print(productExceptSelf([-1, 1, 0, -3, 3]))    # [0, 0, 9, 0, 0]
print(productExceptSelf([0, 0]))               # [0, 0]

## Complexity Analysis

| Approach | Time | Space | Notes |
|----------|------|-------|-------|
| Brute Force | O(n²) | O(n) | Product for each position |
| Left + Right arrays | O(n) | O(n) | Two extra arrays |
| **Left + Running suffix** | **O(n)** | **O(1)*** | **In-place on result** |

*The output array `result` doesn't count as extra space per the problem statement — you must return an array anyway.

**Handles zeros:** If nums = [0, 1, 2]:
- prefix pass:  result = [1, 0, 0]
- suffix pass:  result = [1*2, 0*2, 0*1] = [2, 0, 0]

The zero propagates naturally — no special casing needed.

**Multiple zeros:** If nums = [0, 0, 1], all products are 0 (two zeros → neither side can avoid having a zero). The algorithm handles this correctly without any branches.

## Interview Q&A

**Q: Why is division not allowed?**
A: Division by zero is undefined. If `nums[i] = 0`, you'd divide by zero. The prefix/suffix approach handles zero naturally — no special cases needed.

**Q: What's the extra space in your O(1) solution?**
A: Just two scalar variables — `prefix` and `suffix`. The output array `result` is required (the problem asks for it), so it doesn't count as extra space.

**Q: What happens with two zeros in the array?**
A: Every output value becomes 0. If two elements are 0, every subarray "except self" contains at least one 0. The algorithm computes this correctly without any special handling.

**Q: How is this pattern useful in data engineering?**
A: Running products appear in cumulative calculations — compounding growth rates, probability chains (joint probability of a sequence of events), or building prefix-product lookup tables for O(1) range product queries. Once you have the prefix products array, `product(nums[i..j]) = prefix[j+1] / prefix[i]` — but with division, so pre-compute for zero-safe arrays.

**Q: Can you solve this with a single pass?**
A: No — to know the right product at index `i`, you need to have seen everything to the right. Single-pass isn't possible without extra space. Two passes is the theoretical minimum for this approach.

## The Citi Angle

**Running products in capacity planning:** At Citi, compound growth calculations were common — if a server's CPU grows 2% per week, what is the cumulative utilization after k weeks? This is a prefix product computation.

**More directly:** When computing a rolling efficiency score across a chain of pipeline stages, you multiply stage efficiencies together. If any stage has 0% efficiency (is down), the entire pipeline product is 0. The prefix/suffix decomposition handles this correctly.

**Interview tie-in:** "The 'no division' constraint is a proxy for zero-robustness — and in telemetry data, zeros are common (servers that are off, metrics that aren't reporting). The prefix/suffix pattern handles zeros without special-casing, which matters in production."

## Summary

| | Value |
|--|--|
| **Pattern** | Prefix + Suffix products |
| **Time** | O(n) |
| **Space** | O(1) extra |
| **Key insight** | `result[i] = left_product × right_product` — build each in one pass |
| **Say in interview** | "Decompose into left product and right product. Two passes, O(1) extra space." |

**Template to memorize:**
```python
result = [1] * n
prefix = 1
for i in range(n):
    result[i] = prefix; prefix *= nums[i]

suffix = 1
for i in range(n-1, -1, -1):
    result[i] *= suffix; suffix *= nums[i]
```